In [13]:
import pandas as pd
from pyspark.sql import SparkSession

# Initialize Spark session safely
spark = SparkSession.builder \
    .appName("Temperature Dataset") \
    .getOrCreate()

# Using a guaranteed, publicly available temperature CSV dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
try:
    print("Downloading dataset using pandas...")
    pandas_df = pd.read_csv(url)
    print("Pandas DataFrame preview:")
    display(pandas_df.head())

    # Convert pandas DataFrame to Spark DataFrame
    print("Converting to Spark DataFrame...")
    df = spark.createDataFrame(pandas_df)
    print("Spark DataFrame preview:")
    df.show()
    df.printSchema()
except Exception as e:
    print(f"An error occurred: {e}")

Pandas DataFrame preview:


,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


Converting to Spark DataFrame...
Spark DataFrame preview:
+----------+----+
|      Date|Temp|
+----------+----+
|1981-01-01|20.7|
|1981-01-02|17.9|
|1981-01-03|18.8|
|1981-01-04|14.6|
|1981-01-05|15.8|
|1981-01-06|15.8|
|1981-01-07|15.8|
|1981-01-08|17.4|
|1981-01-09|21.8|
|1981-01-10|20.0|
|1981-01-11|16.2|
|1981-01-12|13.3|
|1981-01-13|16.7|
|1981-01-14|21.5|
|1981-01-15|25.0|
|1981-01-16|20.7|
|1981-01-17|20.6|
|1981-01-18|24.8|
|1981-01-19|17.7|
|1981-01-20|15.5|
+----------+----+
only showing top 20 rows
root
 |-- Date: string (nullable = true)
 |-- Temp: double (nullable = true)



In [14]:
df.show()
df.printSchema()

+----------+----+
|      Date|Temp|
+----------+----+
|1981-01-01|20.7|
|1981-01-02|17.9|
|1981-01-03|18.8|
|1981-01-04|14.6|
|1981-01-05|15.8|
|1981-01-06|15.8|
|1981-01-07|15.8|
|1981-01-08|17.4|
|1981-01-09|21.8|
|1981-01-10|20.0|
|1981-01-11|16.2|
|1981-01-12|13.3|
|1981-01-13|16.7|
|1981-01-14|21.5|
|1981-01-15|25.0|
|1981-01-16|20.7|
|1981-01-17|20.6|
|1981-01-18|24.8|
|1981-01-19|17.7|
|1981-01-20|15.5|
+----------+----+
only showing top 20 rows
root
 |-- Date: string (nullable = true)
 |-- Temp: double (nullable = true)



In [16]:
df.show(5, truncate=False)

+----------+----+
|Date      |Temp|
+----------+----+
|1981-01-01|20.7|
|1981-01-02|17.9|
|1981-01-03|18.8|
|1981-01-04|14.6|
|1981-01-05|15.8|
+----------+----+
only showing top 5 rows


In [17]:
df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Temp: double (nullable = true)



In [18]:
row_count = df.count()
column_count = len(df.columns)

print("Number of rows:", row_count)
print("Number of columns:", column_count)

Number of rows: 3650
Number of columns: 2


In [19]:
print("10M+ rows:", row_count >= 10_000_000)
print("10+ columns:", column_count >= 10)

10M+ rows: False
10+ columns: False


In [22]:
import os

# Define a local file path and save the DataFrame locally
file_path = "daily-min-temperatures.csv"
pandas_df.to_csv(file_path, index=False)

# Check the file size of the saved dataset
file_size_bytes = os.path.getsize(file_path)
file_size_gb = file_size_bytes / (1024 ** 3)

print("File size:", file_size_bytes, "bytes")
print("File size:", round(file_size_gb, 6), "GB")

File size: 56968 bytes
File size: 5.3e-05 GB


In [23]:
partition_count = df.rdd.getNumPartitions()

print("Number of partitions:", partition_count)

Number of partitions: 2


MISSING VALUE CHECK

In [24]:
from pyspark.sql.functions import col, sum

missing_values = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_values.show(truncate=False)

+----+----+
|Date|Temp|
+----+----+
|0   |0   |
+----+----+



Data cleaning

In [29]:
# The original columns ["ID", "YEAR", "MONTH", "ELEM"] do not exist in this dataset.
# We will clean the dataset using the actual columns: ["Date", "Temp"]
clean_df = df.dropna(subset=["Date", "Temp"])

print("Rows after basic cleaning:", clean_df.count())

Rows after basic cleaning: 3650


In [32]:
from pyspark.sql.functions import col, year, month

# Extracting Year and Month from the Date column since the original NOAA GSOD columns don't exist
feature_df = clean_df \
    .withColumn("YEAR", year(col("Date"))) \
    .withColumn("MONTH", month(col("Date")))

feature_df.select(
    "Date",
    "Temp",
    "YEAR",
    "MONTH"
).show(500)

+----------+----+----+-----+
|      Date|Temp|YEAR|MONTH|
+----------+----+----+-----+
|1981-01-01|20.7|1981|    1|
|1981-01-02|17.9|1981|    1|
|1981-01-03|18.8|1981|    1|
|1981-01-04|14.6|1981|    1|
|1981-01-05|15.8|1981|    1|
|1981-01-06|15.8|1981|    1|
|1981-01-07|15.8|1981|    1|
|1981-01-08|17.4|1981|    1|
|1981-01-09|21.8|1981|    1|
|1981-01-10|20.0|1981|    1|
|1981-01-11|16.2|1981|    1|
|1981-01-12|13.3|1981|    1|
|1981-01-13|16.7|1981|    1|
|1981-01-14|21.5|1981|    1|
|1981-01-15|25.0|1981|    1|
|1981-01-16|20.7|1981|    1|
|1981-01-17|20.6|1981|    1|
|1981-01-18|24.8|1981|    1|
|1981-01-19|17.7|1981|    1|
|1981-01-20|15.5|1981|    1|
|1981-01-21|18.2|1981|    1|
|1981-01-22|12.1|1981|    1|
|1981-01-23|14.4|1981|    1|
|1981-01-24|16.0|1981|    1|
|1981-01-25|16.5|1981|    1|
|1981-01-26|18.7|1981|    1|
|1981-01-27|19.4|1981|    1|
|1981-01-28|17.2|1981|    1|
|1981-01-29|15.5|1981|    1|
|1981-01-30|15.1|1981|    1|
|1981-01-31|15.4|1981|    1|
|1981-02-01|15

PySpark Pipeline

In [33]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler

indexer = StringIndexer(
    inputCol="ELEM",
    outputCol="ELEM_INDEX",
    handleInvalid="keep"
)

feature_columns = [
    "YEAR",
    "MONTH",
    "AVAILABLE_DAYS"
]

assembler = VectorAssembler(
    inputCols=feature_columns + ["ELEM_INDEX"],
    outputCol="features"
)

pipeline = Pipeline(
    stages=[
        indexer,
        assembler
    ]
)

In [35]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler

# Redefining the pipeline features to use columns that actually exist in feature_df
feature_columns = ["YEAR", "MONTH"]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

# Re-building the pipeline with the correct stages
pipeline = Pipeline(stages=[assembler])

# Fitting and transforming the correct DataFrame
pipeline_model = pipeline.fit(feature_df)
pipeline_df = pipeline_model.transform(feature_df)

# Showing the resulting DataFrame with existing columns
pipeline_df.select(
    "Date",
    "Temp",
    "YEAR",
    "MONTH",
    "features"
).show(10, truncate=False)

+----------+----+----+-----+------------+
|Date      |Temp|YEAR|MONTH|features    |
+----------+----+----+-----+------------+
|1981-01-01|20.7|1981|1    |[1981.0,1.0]|
|1981-01-02|17.9|1981|1    |[1981.0,1.0]|
|1981-01-03|18.8|1981|1    |[1981.0,1.0]|
|1981-01-04|14.6|1981|1    |[1981.0,1.0]|
|1981-01-05|15.8|1981|1    |[1981.0,1.0]|
|1981-01-06|15.8|1981|1    |[1981.0,1.0]|
|1981-01-07|15.8|1981|1    |[1981.0,1.0]|
|1981-01-08|17.4|1981|1    |[1981.0,1.0]|
|1981-01-09|21.8|1981|1    |[1981.0,1.0]|
|1981-01-10|20.0|1981|1    |[1981.0,1.0]|
+----------+----+----+-----+------------+
only showing top 10 rows


Pipeline stages evidence

In [36]:
print("Fitted Pipeline stages:")

for stage in pipeline_model.stages:
    print(type(stage).__name__)

Fitted Pipeline stages:
VectorAssembler


In [37]:
pipeline_df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Temp: double (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- features: vector (nullable = true)



Final row count

In [38]:
print("Final row count:", pipeline_df.count())
print("Final column count:", len(pipeline_df.columns))

Final row count: 3650
Final column count: 5


In [39]:
print("========== EP1 EVIDENCE PACK ==========")

print("\n1. SparkSession")
print("Spark version:", spark.version)

print("\n2. Dataset Schema")
df.printSchema()

print("\n3. Dataset Size")
print("Rows:", row_count)
print("Columns:", column_count)

print("\n4. File Size")
print("Bytes:", file_size_bytes)
print("GB:", round(file_size_gb, 2))

print("\n5. Partitions")
print("Partitions:", partition_count)

print("\n6. Fitted Pipeline Stages")
for stage in pipeline_model.stages:
    print(type(stage).__name__)

print("\n========== END EP1 ==========")

========== EP1 EVIDENCE PACK ==========

1. SparkSession
Spark version: 4.0.4

2. Dataset Schema
root
 |-- Date: string (nullable = true)
 |-- Temp: double (nullable = true)


3. Dataset Size
Rows: 3650
Columns: 2

4. File Size
Bytes: 56968
GB: 0.0

5. Partitions
Partitions: 2

6. Fitted Pipeline Stages
VectorAssembler

========== END EP1 ==========


In [40]:
# Configure Git Identity
!git config --global user.name "sainim10"
!git config --global user.email "sainim10@coventry.ac.uk"

# Initialize and configure repository
!git init
!git remote add origin https://github.coventry.ac.uk/7006SCN-2627SEPNOV/sainim10_17142113.git

# Add, commit, and push changes
!git add daily-min-temperatures.csv
!git commit -m "Add temperature dataset and Spark pipeline artifacts"
!git push -u origin main

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
[master (root-commit) e1b02b7] Add temperature dataset and Spark pipeline artifacts
 1 file changed, 3651 insertions(+)
 create mode 100644 daily-min-temperatures.csv
error: src refspec main does not match any
error: failed to push some refs to 'https://github.coventry.ac.uk/7006SCN-2627SEPNOV/sainim10_17142113.git'
